# DataGuard: NYC 311 Missing Value Report

## Notebook Purpose

This notebook is the third notebook in the DataGuard project.

The goal of Day 3 is to analyze missing values in the NYC 311 dataset and create the first DataGuard data quality report.

This notebook focuses on:

- Identifying columns with missing values
- Calculating missing counts
- Calculating missing percentages
- Classifying missing values by business importance
- Exporting a missing value report as a CSV file

## Analytical Approach

Missing value analysis is not just about finding `NaN` values.

The goal is to understand which missing values are expected, which are problematic, and which should be flagged for later cleaning, validation, or business review.

In [1]:
import pandas as pd
from pathlib import Path

pd.set_option("display.max_columns", None)

print("Imports complete.")

Imports complete.


In [2]:
project_root = Path.cwd()

# If this notebook is running from the notebooks folder, move one level up to DataGuard
if project_root.name == "notebooks":
    project_root = project_root.parent

data_path = project_root / "data" / "nyc_311_raw_100k.csv"

df = pd.read_csv(data_path)

print("Dataset loaded successfully.")
print("Rows and columns:", df.shape)

df.head()

Dataset loaded successfully.
Rows and columns: (100000, 44)


,unique_key,created_date,closed_date,agency,agency_name,complaint_type,descriptor,descriptor_2,location_type,incident_zip,incident_address,street_name,cross_street_1,cross_street_2,intersection_street_1,intersection_street_2,address_type,city,landmark,facility_type,status,due_date,resolution_description,resolution_action_updated_date,community_board,council_district,police_precinct,bbl,borough,x_coordinate_state_plane,y_coordinate_state_plane,open_data_channel_type,park_facility_name,park_borough,vehicle_type,taxi_company_borough,taxi_pick_up_location,bridge_highway_name,bridge_highway_direction,road_ramp,bridge_highway_segment,latitude,longitude,location
0,69486831,2026-06-26T02:06:32.000,NaN,NYPD,New York City Police Department,Noise - Commercial,Loud Music/Party,NaN,Club/Bar/Restaurant,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,In Progress,NaN,NaN,NaN,0 Unspecified,NaN,Unspecified,NaN,Unspecified,991542.0,189307.0,ONLINE,Unspecified,Unspecified,NaN,NaN,NaN,NaN,NaN,NaN,NaN,40.686278,-73.973707,POINT (-73.9737071634 40.686277751362)
1,69492768,2026-06-26T02:06:05.000,NaN,TLC,Taxi and Limousine Commission,For Hire Vehicle Complaint,Driver Complaint - Non Passenger,Unsafe Driving,Street,11211.0,180 UNION AVENUE,UNION AVENUE,BROADWAY,JOHNSON AVENUE,BROADWAY,JOHNSON AVENUE,ADDRESS,BROOKLYN,UNION AVENUE,NaN,In Progress,NaN,NaN,NaN,01 BROOKLYN,34.0,Precinct 90,3.030580e+09,BROOKLYN,998041.0,196408.0,ONLINE,Unspecified,BROOKLYN,NaN,NaN,"180 UNION AVENUE, BROOKLYN, NY, 11211",NaN,NaN,NaN,NaN,40.705761,-73.950259,POINT (-73.950259119641 40.705760603484)
2,69482343,2026-06-26T02:05:24.000,NaN,NYPD,New York City Police Department,Noise - Residential,Loud Music/Party,NaN,Residential Building/House,10013.0,118 MULBERRY STREET,MULBERRY STREET,CANAL STREET,HESTER STREET,CANAL STREET,HESTER STREET,ADDRESS,NEW YORK,MULBERRY STREET,NaN,In Progress,NaN,NaN,NaN,02 MANHATTAN,1.0,Precinct 5,1.002050e+09,MANHATTAN,984753.0,200765.0,ONLINE,Unspecified,MANHATTAN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,40.717730,-73.998185,POINT (-73.998185472064 40.717730264942)
3,69488306,2026-06-26T02:03:49.000,NaN,NYPD,New York City Police Department,Noise - Street/Sidewalk,Loud Music/Party,NaN,Street/Sidewalk,11249.0,96 WYTHE AVENUE,WYTHE AVENUE,NORTH 11 STREET,NORTH 10 STREET,NORTH 11 STREET,NORTH 10 STREET,ADDRESS,BROOKLYN,WYTHE AVENUE,NaN,In Progress,NaN,NaN,NaN,01 BROOKLYN,33.0,Precinct 94,3.022950e+09,BROOKLYN,995800.0,202137.0,ONLINE,Unspecified,BROOKLYN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,40.721489,-73.958332,POINT (-73.958332047132 40.721488549007)
4,69482692,2026-06-26T02:03:29.000,NaN,NYPD,New York City Police Department,Illegal Parking,Blocked Crosswalk,NaN,Street/Sidewalk,11421.0,85 STREET,85 STREET,85 STREET,88 ROAD,85 STREET,88 ROAD,INTERSECTION,NaN,NaN,NaN,In Progress,NaN,NaN,NaN,09 QUEENS,32.0,Precinct 102,NaN,QUEENS,1023600.0,190652.0,ONLINE,Unspecified,QUEENS,NaN,NaN,NaN,NaN,NaN,NaN,NaN,40.689885,-73.858108,POINT (-73.858107579574 40.689884995274)


## Understanding Missing Values

Missing values are blank or unavailable values in a dataset. In pandas, these often appear as `NaN`.

For DataGuard, the goal is not to immediately delete or fill missing values. The goal is to first detect, measure, and understand them.

A missing value can mean different things depending on the column:

- It may be a real data quality issue
- It may be expected because the field does not apply to that record
- It may indicate an open or incomplete service request
- It may need to be flagged for later review

In [ ]:
# Check whether each value is missing
df.isnull().head1212()

,unique_key,created_date,closed_date,agency,agency_name,complaint_type,descriptor,descriptor_2,location_type,incident_zip,incident_address,street_name,cross_street_1,cross_street_2,intersection_street_1,intersection_street_2,address_type,city,landmark,facility_type,status,due_date,resolution_description,resolution_action_updated_date,community_board,council_district,police_precinct,bbl,borough,x_coordinate_state_plane,y_coordinate_state_plane,open_data_channel_type,park_facility_name,park_borough,vehicle_type,taxi_company_borough,taxi_pick_up_location,bridge_highway_name,bridge_highway_direction,road_ramp,bridge_highway_segment,latitude,longitude,location
0,False,False,True,False,False,False,False,True,False,True,True,True,True,True,True,True,True,True,True,True,False,True,True,True,False,True,False,True,False,False,False,False,False,False,True,True,True,True,True,True,True,False,False,False
1,False,False,True,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,True,False,True,True,True,False,False,False,False,False,False,False,False,False,False,True,True,False,True,True,True,True,False,False,False
2,False,False,True,False,False,False,False,True,False,False,False,False,False,False,False,False,False,False,False,True,False,True,True,True,False,False,False,False,False,False,False,False,False,False,True,True,True,True,True,True,True,False,False,False
3,False,False,True,False,False,False,False,True,False,False,False,False,False,False,False,False,False,False,False,True,False,True,True,True,False,False,False,False,False,False,False,False,False,False,True,True,True,True,True,True,True,False,False,False
4,False,False,True,False,False,False,False,True,False,False,False,False,False,False,False,False,False,True,True,True,False,True,True,True,False,False,False,True,False,False,False,False,False,False,True,True,True,True,True,True,True,False,False,False


In [4]:
# Count missing values in each column
missing_counts = df.isnull().sum()

missing_counts

unique_key                            0
created_date                          0
closed_date                       25095
agency                                0
agency_name                           0
complaint_type                        0
descriptor                         2397
descriptor_2                      66778
location_type                     13161
incident_zip                        811
incident_address                   3719
street_name                        3719
cross_street_1                    21239
cross_street_2                    21268
intersection_street_1             25202
intersection_street_2             25207
address_type                        537
city                               5999
landmark                          33400
facility_type                     99775
status                                0
due_date                          99574
resolution_description            11140
resolution_action_updated_date    10269
community_board                       0


In [5]:
# Calculate missing percentage for each column
missing_percentages = (df.isnull().sum() / len(df)) * 100

missing_percentages

unique_key                         0.000
created_date                       0.000
closed_date                       25.095
agency                             0.000
agency_name                        0.000
complaint_type                     0.000
descriptor                         2.397
descriptor_2                      66.778
location_type                     13.161
incident_zip                       0.811
incident_address                   3.719
street_name                        3.719
cross_street_1                    21.239
cross_street_2                    21.268
intersection_street_1             25.202
intersection_street_2             25.207
address_type                       0.537
city                               5.999
landmark                          33.400
facility_type                     99.775
status                             0.000
due_date                          99.574
resolution_description            11.140
resolution_action_updated_date    10.269
community_board 

In [6]:
# Create a missing value report table
missing_report = pd.DataFrame({
    "column_name": df.columns,
    "data_type": df.dtypes.values,
    "missing_count": df.isnull().sum().values,
    "missing_percentage": ((df.isnull().sum() / len(df)) * 100).values,
    "non_null_count": df.notnull().sum().values
})

# Sort by highest missing percentage
missing_report = missing_report.sort_values(
    by="missing_percentage",
    ascending=False
)

missing_report

,column_name,data_type,missing_count,missing_percentage,non_null_count
35,taxi_company_borough,object,99922,99.922,78
19,facility_type,object,99775,99.775,225
38,bridge_highway_direction,object,99744,99.744,256
39,road_ramp,object,99743,99.743,257
21,due_date,object,99574,99.574,426
40,bridge_highway_segment,object,99544,99.544,456
37,bridge_highway_name,object,99538,99.538,462
36,taxi_pick_up_location,object,98753,98.753,1247
34,vehicle_type,object,94573,94.573,5427
7,descriptor_2,object,66778,66.778,33222


In [7]:
# Filter report to only columns with missing values
columns_with_missing_values = missing_report[missing_report["missing_count"] > 0]

columns_with_missing_values

,column_name,data_type,missing_count,missing_percentage,non_null_count
35,taxi_company_borough,object,99922,99.922,78
19,facility_type,object,99775,99.775,225
38,bridge_highway_direction,object,99744,99.744,256
39,road_ramp,object,99743,99.743,257
21,due_date,object,99574,99.574,426
40,bridge_highway_segment,object,99544,99.544,456
37,bridge_highway_name,object,99538,99.538,462
36,taxi_pick_up_location,object,98753,98.753,1247
34,vehicle_type,object,94573,94.573,5427
7,descriptor_2,object,66778,66.778,33222


In [8]:
print("Total columns:", missing_report.shape[0])
print("Columns with missing values:", columns_with_missing_values.shape[0])
print("Columns with no missing values:", missing_report.shape[0] - columns_with_missing_values.shape[0])

Total columns: 44
Columns with missing values: 33
Columns with no missing values: 11


### Observation: Columns With Missing Values

Out of 44 total columns, 33 columns contain at least one missing value. The remaining 11 columns are fully complete in this 100,000-row sample.

This shows that missing values are common in the NYC 311 dataset. However, missingness does not automatically mean the data is bad. Some missing values may be expected because certain fields only apply to specific complaint types, while other missing values may require review during the data quality process.

## Classifying Missing Value Severity

To make the missing value report more useful, I will classify columns into severity levels based on the percentage of missing values.

This gives a quick way to identify which columns need the most attention during later data quality checks.

In [9]:
def classify_missing_severity(missing_percentage):
    if missing_percentage == 0:
        return "No Missing Values"
    elif missing_percentage < 5:
        return "Low"
    elif missing_percentage < 25:
        return "Medium"
    elif missing_percentage < 75:
        return "High"
    else:
        return "Very High"

missing_report["missing_severity"] = missing_report["missing_percentage"].apply(classify_missing_severity)

missing_report.head(15)

,column_name,data_type,missing_count,missing_percentage,non_null_count,missing_severity
35,taxi_company_borough,object,99922,99.922,78,Very High
19,facility_type,object,99775,99.775,225,Very High
38,bridge_highway_direction,object,99744,99.744,256,Very High
39,road_ramp,object,99743,99.743,257,Very High
21,due_date,object,99574,99.574,426,Very High
40,bridge_highway_segment,object,99544,99.544,456,Very High
37,bridge_highway_name,object,99538,99.538,462,Very High
36,taxi_pick_up_location,object,98753,98.753,1247,Very High
34,vehicle_type,object,94573,94.573,5427,Very High
7,descriptor_2,object,66778,66.778,33222,High


## Classifying Business Importance

Missing percentage alone does not fully explain whether a column is important.

Some columns may have very high missing percentages because they only apply to specific complaint types. Other columns may have lower missing percentages but still be important because they affect core analysis.

To make the report more useful, I will classify columns by business importance.

In [10]:
critical_columns = [
    "unique_key",
    "created_date",
    "agency",
    "agency_name",
    "complaint_type",
    "status"
]

important_columns = [
    "closed_date",
    "borough",
    "city",
    "incident_zip",
    "latitude",
    "longitude",
    "location",
    "resolution_description",
    "resolution_action_updated_date"
]

conditional_columns = [
    "taxi_company_borough",
    "taxi_pick_up_location",
    "vehicle_type",
    "bridge_highway_name",
    "bridge_highway_direction",
    "road_ramp",
    "bridge_highway_segment",
    "facility_type"
]

def classify_business_importance(column_name):
    if column_name in critical_columns:
        return "Critical"
    elif column_name in important_columns:
        return "Important"
    elif column_name in conditional_columns:
        return "Conditional"
    else:
        return "Optional / Review"

missing_report["business_importance"] = missing_report["column_name"].apply(classify_business_importance)

missing_report.head(15)

,column_name,data_type,missing_count,missing_percentage,non_null_count,missing_severity,business_importance
35,taxi_company_borough,object,99922,99.922,78,Very High,Conditional
19,facility_type,object,99775,99.775,225,Very High,Conditional
38,bridge_highway_direction,object,99744,99.744,256,Very High,Conditional
39,road_ramp,object,99743,99.743,257,Very High,Conditional
21,due_date,object,99574,99.574,426,Very High,Optional / Review
40,bridge_highway_segment,object,99544,99.544,456,Very High,Conditional
37,bridge_highway_name,object,99538,99.538,462,Very High,Conditional
36,taxi_pick_up_location,object,98753,98.753,1247,Very High,Conditional
34,vehicle_type,object,94573,94.573,5427,Very High,Conditional
7,descriptor_2,object,66778,66.778,33222,High,Optional / Review


In [12]:
def recommend_missing_action(row):
    column_name = row["column_name"]
    missing_count = row["missing_count"]
    business_importance = row["business_importance"]
    missing_severity = row["missing_severity"]
    
    if missing_count == 0:
        return "No action needed"
    
    elif business_importance == "Critical":
        return "Flag as critical data quality issue"
    
    elif business_importance == "Important" and column_name == "closed_date":
        return "Review with status; missing may be expected for open requests"
    
    elif business_importance == "Important":
        return "Review and consider flagging or filling with UNKNOWN if appropriate"
    
    elif business_importance == "Conditional":
        return "Likely expected missingness; validate against related complaint types"
    
    else:
        return "Optional field; review before deciding to drop, fill, or ignore"

missing_report["recommended_action"] = missing_report.apply(recommend_missing_action, axis=1)

missing_report.head(15)

,column_name,data_type,missing_count,missing_percentage,non_null_count,missing_severity,business_importance,recommended_action
35,taxi_company_borough,object,99922,99.922,78,Very High,Conditional,Likely expected missingness; validate against ...
19,facility_type,object,99775,99.775,225,Very High,Conditional,Likely expected missingness; validate against ...
38,bridge_highway_direction,object,99744,99.744,256,Very High,Conditional,Likely expected missingness; validate against ...
39,road_ramp,object,99743,99.743,257,Very High,Conditional,Likely expected missingness; validate against ...
21,due_date,object,99574,99.574,426,Very High,Optional / Review,Optional field; review before deciding to drop...
40,bridge_highway_segment,object,99544,99.544,456,Very High,Conditional,Likely expected missingness; validate against ...
37,bridge_highway_name,object,99538,99.538,462,Very High,Conditional,Likely expected missingness; validate against ...
36,taxi_pick_up_location,object,98753,98.753,1247,Very High,Conditional,Likely expected missingness; validate against ...
34,vehicle_type,object,94573,94.573,5427,Very High,Conditional,Likely expected missingness; validate against ...
7,descriptor_2,object,66778,66.778,33222,High,Optional / Review,Optional field; review before deciding to drop...


In [13]:
# Count columns by missing severity level
missing_report["missing_severity"].value_counts()

missing_severity
Low                  12
No Missing Values    11
Very High             9
Medium                7
High                  5
Name: count, dtype: int64

### Observation: Missing Severity Summary

The missing value severity summary shows that 11 columns have no missing values, while 33 columns contain at least some missing data.

There are 9 columns with `Very High` missingness and 5 columns with `High` missingness. These columns need further review, but high missingness does not automatically mean the column is bad. Some high-missing columns may be conditional fields that only apply to specific complaint types.

This confirms that missing values are a major data quality theme in the NYC 311 dataset and should be tracked as part of DataGuard.

In [14]:
# Count columns by business importance
missing_report["business_importance"].value_counts()

business_importance
Optional / Review    21
Important             9
Conditional           8
Critical              6
Name: count, dtype: int64

### Observation: Business Importance Summary

The missing value report classifies 6 columns as `Critical`, 9 columns as `Important`, 8 columns as `Conditional`, and 21 columns as `Optional / Review`.

The `Critical` columns are required for basic analysis and should have very little or no missingness. The `Important` columns affect common analyses such as location analysis, resolution tracking, and reporting. The `Conditional` columns may only apply to specific complaint types, so missingness in those fields may be expected.

This classification helps separate missing values that are truly concerning from missing values that may be normal for the dataset structure.

In [15]:
# Review critical columns
critical_missing_report = missing_report[
    missing_report["business_importance"] == "Critical"
]

critical_missing_report

,column_name,data_type,missing_count,missing_percentage,non_null_count,missing_severity,business_importance,recommended_action
0,unique_key,int64,0,0.0,100000,No Missing Values,Critical,No action needed
1,created_date,object,0,0.0,100000,No Missing Values,Critical,No action needed
3,agency,object,0,0.0,100000,No Missing Values,Critical,No action needed
5,complaint_type,object,0,0.0,100000,No Missing Values,Critical,No action needed
4,agency_name,object,0,0.0,100000,No Missing Values,Critical,No action needed
20,status,object,0,0.0,100000,No Missing Values,Critical,No action needed


### Observation: Critical Columns

All critical columns have 0 missing values in this sample.

The complete critical fields are `unique_key`, `created_date`, `agency`, `agency_name`, `complaint_type`, and `status`.

This is a positive data quality finding because the core fields needed for identifying, timing, categorizing, routing, and tracking 311 service requests are complete in the sample.

In [16]:
# Review important columns
important_missing_report = missing_report[
    missing_report["business_importance"] == "Important"
]

important_missing_report

,column_name,data_type,missing_count,missing_percentage,non_null_count,missing_severity,business_importance,recommended_action
2,closed_date,object,25095,25.095,74905,High,Important,Review with status; missing may be expected fo...
22,resolution_description,object,11140,11.140,88860,Medium,Important,Review and consider flagging or filling with U...
23,resolution_action_updated_date,object,10269,10.269,89731,Medium,Important,Review and consider flagging or filling with U...
17,city,object,5999,5.999,94001,Medium,Important,Review and consider flagging or filling with U...
43,location,object,2035,2.035,97965,Low,Important,Review and consider flagging or filling with U...
42,longitude,float64,2035,2.035,97965,Low,Important,Review and consider flagging or filling with U...
41,latitude,float64,2035,2.035,97965,Low,Important,Review and consider flagging or filling with U...
9,incident_zip,float64,811,0.811,99189,Low,Important,Review and consider flagging or filling with U...
28,borough,object,0,0.000,100000,No Missing Values,Important,No action needed


### Observation: Important Columns

The important columns show mixed levels of missingness.

The `closed_date` column has 25,095 missing values, or 25.095% missingness. This is important because `closed_date` is needed for resolution-time analysis. However, missing `closed_date` values may be expected for requests that are still `Open` or `In Progress`.

Location-related fields such as `latitude`, `longitude`, and `location` have 2.035% missingness, while `incident_zip` has 0.811% missingness. These are relatively low missing percentages, but they may still affect location-based analysis.

The `borough` column has 0 missing values, which is a positive finding because borough-level analysis is one of the main ways to summarize NYC 311 complaint patterns.

The `city` column has 5.999% missingness and may need review later, especially because earlier analysis showed that this column contains a mix of borough names and neighborhood/location names.

In [17]:
# Check statuses for records with missing closed_date
missing_closed_date_status = df[df["closed_date"].isnull()]["status"].value_counts()

missing_closed_date_status

status
Open           12338
In Progress    12183
Started          448
Assigned          99
Pending           27
Name: count, dtype: int64

### Observation: Missing Closed Dates by Status

All records with missing `closed_date` values have non-closed statuses, such as `Open`, `In Progress`, `Started`, `Assigned`, or `Pending`.

This suggests that missing `closed_date` values are expected for these records because the service requests have not been closed yet.

This is an important DataGuard finding: a missing value is not automatically a data quality issue. The meaning of the missing value depends on the business context and related fields.

In [18]:
# Review conditional columns
conditional_missing_report = missing_report[
    missing_report["business_importance"] == "Conditional"
]

conditional_missing_report

,column_name,data_type,missing_count,missing_percentage,non_null_count,missing_severity,business_importance,recommended_action
35,taxi_company_borough,object,99922,99.922,78,Very High,Conditional,Likely expected missingness; validate against ...
19,facility_type,object,99775,99.775,225,Very High,Conditional,Likely expected missingness; validate against ...
38,bridge_highway_direction,object,99744,99.744,256,Very High,Conditional,Likely expected missingness; validate against ...
39,road_ramp,object,99743,99.743,257,Very High,Conditional,Likely expected missingness; validate against ...
40,bridge_highway_segment,object,99544,99.544,456,Very High,Conditional,Likely expected missingness; validate against ...
37,bridge_highway_name,object,99538,99.538,462,Very High,Conditional,Likely expected missingness; validate against ...
36,taxi_pick_up_location,object,98753,98.753,1247,Very High,Conditional,Likely expected missingness; validate against ...
34,vehicle_type,object,94573,94.573,5427,Very High,Conditional,Likely expected missingness; validate against ...


In [19]:
# Check complaint types where taxi_pick_up_location is present
df[df["taxi_pick_up_location"].notnull()]["complaint_type"].value_counts().head(10)

complaint_type
For Hire Vehicle Complaint    689
Taxi Complaint                305
Lost Property                 183
For Hire Vehicle Report        35
Taxi Report                    22
Green Taxi Complaint            7
Taxi Compliment                 2
Found Property                  2
Green Taxi Report               1
FHV Licensee Complaint          1
Name: count, dtype: int64

### Observation: Conditional Taxi Field

The `taxi_pick_up_location` field is mostly populated for taxi and for-hire-vehicle related complaint types, such as `For Hire Vehicle Complaint`, `Taxi Complaint`, and `For Hire Vehicle Report`.

This supports the idea that missing values in taxi-related columns are expected for most non-taxi complaint records.

Therefore, high missingness in `taxi_pick_up_location` should not automatically be treated as a data quality failure. Instead, it should be validated conditionally based on complaint type.

In [20]:
# Check complaint types where bridge_highway_name is present
df[df["bridge_highway_name"].notnull()]["complaint_type"].value_counts().head(10)

complaint_type
Highway Condition             118
Homeless Person Assistance    112
Panhandling                   100
Encampment                     75
Taxi Complaint                 15
Traffic                        13
For Hire Vehicle Complaint      8
Bridge Condition                7
Dead Animal                     5
Illegal Fireworks               4
Name: count, dtype: int64

In [21]:
# Final missing value report columns for export
final_missing_report = missing_report[
    [
        "column_name",
        "data_type",
        "non_null_count",
        "missing_count",
        "missing_percentage",
        "missing_severity",
        "business_importance",
        "recommended_action"
    ]
]

final_missing_report

,column_name,data_type,non_null_count,missing_count,missing_percentage,missing_severity,business_importance,recommended_action
35,taxi_company_borough,object,78,99922,99.922,Very High,Conditional,Likely expected missingness; validate against ...
19,facility_type,object,225,99775,99.775,Very High,Conditional,Likely expected missingness; validate against ...
38,bridge_highway_direction,object,256,99744,99.744,Very High,Conditional,Likely expected missingness; validate against ...
39,road_ramp,object,257,99743,99.743,Very High,Conditional,Likely expected missingness; validate against ...
21,due_date,object,426,99574,99.574,Very High,Optional / Review,Optional field; review before deciding to drop...
40,bridge_highway_segment,object,456,99544,99.544,Very High,Conditional,Likely expected missingness; validate against ...
37,bridge_highway_name,object,462,99538,99.538,Very High,Conditional,Likely expected missingness; validate against ...
36,taxi_pick_up_location,object,1247,98753,98.753,Very High,Conditional,Likely expected missingness; validate against ...
34,vehicle_type,object,5427,94573,94.573,Very High,Conditional,Likely expected missingness; validate against ...
7,descriptor_2,object,33222,66778,66.778,High,Optional / Review,Optional field; review before deciding to drop...


In [22]:
# Final missing value report columns for export
final_missing_report = missing_report[
    [
        "column_name",
        "data_type",
        "non_null_count",
        "missing_count",
        "missing_percentage",
        "missing_severity",
        "business_importance",
        "recommended_action"
    ]
]

final_missing_report

,column_name,data_type,non_null_count,missing_count,missing_percentage,missing_severity,business_importance,recommended_action
35,taxi_company_borough,object,78,99922,99.922,Very High,Conditional,Likely expected missingness; validate against ...
19,facility_type,object,225,99775,99.775,Very High,Conditional,Likely expected missingness; validate against ...
38,bridge_highway_direction,object,256,99744,99.744,Very High,Conditional,Likely expected missingness; validate against ...
39,road_ramp,object,257,99743,99.743,Very High,Conditional,Likely expected missingness; validate against ...
21,due_date,object,426,99574,99.574,Very High,Optional / Review,Optional field; review before deciding to drop...
40,bridge_highway_segment,object,456,99544,99.544,Very High,Conditional,Likely expected missingness; validate against ...
37,bridge_highway_name,object,462,99538,99.538,Very High,Conditional,Likely expected missingness; validate against ...
36,taxi_pick_up_location,object,1247,98753,98.753,Very High,Conditional,Likely expected missingness; validate against ...
34,vehicle_type,object,5427,94573,94.573,Very High,Conditional,Likely expected missingness; validate against ...
7,descriptor_2,object,33222,66778,66.778,High,Optional / Review,Optional field; review before deciding to drop...


In [26]:
# Export final missing value report to the reports folder
reports_folder = project_root / "reports"
reports_folder.mkdir(exist_ok=True)

report_path = reports_folder / "missing_value_report_day3.csv"

final_missing_report.to_csv(report_path, index=False)

print("Missing value report saved to:", report_path)

Missing value report saved to: c:\Users\samee\DataGuard\reports\missing_value_report_day3.csv


In [27]:
# Verify exported report exists
report_path.exists()

True

# Day 3 Summary

In this notebook, I created the first DataGuard data quality report focused on missing values in the NYC 311 dataset.

## Work Completed

- Loaded the 100,000-row NYC 311 dataset sample
- Used `isnull()` to detect missing values
- Calculated missing counts for each column
- Calculated missing percentages for each column
- Created a structured missing value report table
- Classified missing values by percentage-based severity
- Classified columns by business importance
- Added recommended actions for future handling
- Reviewed critical, important, and conditional fields
- Validated that missing `closed_date` values are associated with non-closed request statuses
- Validated examples of conditional missingness in taxi and bridge/highway fields
- Exported the final missing value report as `reports/missing_value_report_day3.csv`

## Key Findings

- 33 out of 44 columns contain at least one missing value.
- 11 columns have no missing values.
- All critical columns have 0 missing values, including `unique_key`, `created_date`, `agency`, `agency_name`, `complaint_type`, and `status`.
- `closed_date` has 25.095% missingness, but all missing `closed_date` records have non-closed statuses such as `Open`, `In Progress`, `Started`, `Assigned`, or `Pending`.
- Several highly missing columns are conditional fields, such as taxi-related and bridge/highway-related columns.
- High missingness does not automatically mean a field is bad. Missingness must be interpreted based on business context.

## Output Created

The missing value report was saved to:

`reports/missing_value_report_day3.csv`

## Next Step

The next step is Day 4: detecting additional data issues such as duplicates, outliers, wrong data types, and inconsistent formatting.